# Rule: **build_industrial_energy_demand_per_node_today**


**Description**

This rule distributes national energy demand across nodes (bus regions). This is done by applying distribution keys with values between 0 and 1. The resulting outputs represent industrial energy demand (TWh/a) at the nodal level.

**Inputs**

- resources/{prefix}/{name}/`industrial_distribution_key_base_s_{clusters}.csv`
- resources/{prefix}/{name}/`industrial_energy_demand_per_country_today.csv`

**Outputs**

- resources/{prefix}/{name}/`industrial_energy_demand_today_base_s_{clusters}.csv`

In [ ]:
######################################## Parameters

### Run
prefix = ''
name = ''

### Network
clusters = '' # number of clusters or 'adm'

In [ ]:
##### Imports
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import os 
import sys
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp

##### Read params.yaml
params = xp.read_params('../params.yaml')

##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

##### Region files
region_tag = f"base_s_{clusters}"
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)

##### Set options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## `industrial_energy_demand_today_base_s_{clusters}.csv`  
Load the file and preview its content.

In [ ]:
file = f"industrial_energy_demand_today_base_s_{clusters}.csv"

ind_energy_nodal = xp.load_file_csv(
    params,
    file,
    prefix=prefix,
    name=name,
    location="resources",
)

# Use the first column (country code) as DataFrame index
ind_energy_nodal = ind_energy_nodal.set_index(ind_energy_nodal.columns[0])

ind_energy_nodal.head()

See today's industrial energy demand for a specific country

In [ ]:
# Select the country
country_code = "ES"

ind_energy_nodal_country = ind_energy_nodal[
    ind_energy_nodal.index.str.startswith(country_code)
]

ind_energy_nodal_country

What is the spatial distribution of today's energy demand?

In [ ]:
#### Layout
fig, ax = plt.subplots(figsize=(15, 15))

# Merge with regions
gdf = gdf_regions_onshore.merge(
    ind_energy_nodal_country,
    left_on="name",
    right_index=True,
    how="left"
)

gdf_regions_onshore.plot(
    ax=ax,
    color="white",
    edgecolor="black",
    linewidth=0.7,
)

# Colour dictionary
carrier_colors = {
    "biomass": "#2ca02c",
    "electricity": "#1f77b4",
    "heat": "#ffbf00",
    "gas": "#ff7f0e",
    "liquid": "#d62728",
    "solid": "#7f7f7f",
    "hydrogen": "#17becf",
    "waste": "#9467bd",
}

carriers = [c for c in carrier_colors.keys() if c in gdf.columns]

colors = [carrier_colors[c] for c in carriers]

################ Adjust the size of the pie charts
radius_scale = 1

# Total demand calculations for relative pie chart sizes
total_demand = gdf[carriers].sum(axis=1)
max_demand = total_demand.max()

############# Plot
for i, row in gdf.iterrows():
    if pd.isna(total_demand[i]) or total_demand[i] <= 0:
        continue

    sizes = row[carriers].values.astype(float)
    sizes = np.where(sizes < 0, 0, sizes)

    if sizes.sum() == 0:
        continue

    x = row.geometry.centroid.x
    y = row.geometry.centroid.y

    size = radius_scale * np.sqrt(total_demand[i] / max_demand)

    axins = inset_axes(
        ax,
        width=size,
        height=size,
        loc="center",
        bbox_to_anchor=(x, y),
        bbox_transform=ax.transData,
        borderpad=0,
    )

    axins.pie(
        sizes,
        colors=colors,
        wedgeprops=dict(linewidth=0),
    )
    axins.axis("off")

####### Legend
legend_elements = [
    Line2D(
        [0], [0],
        marker="o",
        color="w",
        label=carrier,
        markerfacecolor=carrier_colors[carrier],
        markersize=8
    )
    for carrier in carriers
]

ax.legend(
    handles=legend_elements,
    title="Energy carrier",
    fontsize=12,
    title_fontsize=12,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

ax.set_aspect("equal")
ax.axis("off")

plt.tight_layout(rect=[0, 0, 0.85, 1])

##### Title
ax.set_title(
    "Industrial energy demand by carrier (today) [TWh/a]",
    fontsize=16
)

plt.show()